# Seq2Seq DPO Fine-Tuning mit Hugging Face DPOTrainer
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook implementiert das DPO-Training des Übersetzungsmodells mit der offiziellen `trl` (Transformer Reinforcement Learning) Library von Hugging Face.

Ablauf:
1. **SFT Baseline laden** (von `results/models/best_sft_w05_w05_filtered_temp.pt`)
2. **Präferenzdatensatz erstellen (Offline DPO)**: Wir generieren für unsere Trainingsdaten Übersetzungskandidaten mit dem SFT-Modell und bewerten diese mit der **Composite Reward-Funktion** (Synthetischer Einfachheits-Regressor + SBERT Semantischer Erhalt).
3. **Hugging Face DPOTrainer konfigurieren & ausführen** – Training des Modells mit dem offiziellen DPO-Verlust.
4. **Evaluierung auf dem unabhängigen Lebenshilfe-Datensatz**.

In [1]:
import os
import sys

# Arbeitsverzeichnis auf das Root-Verzeichnis des Repositories setzen
while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")

print("Aktuelles Arbeitsverzeichnis:", os.getcwd())

Aktuelles Arbeitsverzeichnis: /home/fiete/master-thesis


In [2]:
# ==============================================================================
# ZENTRALE KONFIGURATION & PARAMS
# ==============================================================================
LH_DATASET_PATH = "data/lebenshilfe/lebenshilfe_dataset.json"
CORPUS_CSV_PATH = "data/analysis/corpus_master.csv"
OUTPUT_DIR = "results/models/seq2seq_dpo_w05_w05_filtered_trl"
SFT_MODEL_TEMP_PATH = "results/models/2_sft_filtered.pt"
SYNTHETIC_MODEL_PATH = "results/models/bilstm_synthetic_regression.pt"
SYNTHETIC_VOCAB_PATH = "data/vocabs/synthetic_vocab.json"

MIN_SIM = 0.80
MAX_SIM = 0.98

W_STYLE = 0.5
W_SEM = 0.5

MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256
MODEL_NAME = "facebook/mbart-large-50"


## 1. Setup & Importe

In [3]:
import torch
import random
import numpy as np
import pandas as pd
import json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import spacy
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from trl import DPOConfig, DPOTrainer
from datasets import Dataset as HFDataset
from sentence_transformers import SentenceTransformer, util

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Nutze Device: {DEVICE}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Globaler Seed auf {seed} gesetzt.")

set_seed(42)

Nutze Device: cuda
Globaler Seed auf 42 gesetzt.


## 2. Daten laden & präparieren

In [4]:
def load_filtered_corpus(csv_path=CORPUS_CSV_PATH, min_sim=MIN_SIM, max_sim=MAX_SIM):
    df = pd.read_csv(csv_path)
    filtered_df = df[
        (df["semantic_similarity_8192"] >= min_sim) & 
        (df["semantic_similarity_8192"] <= max_sim)
    ]
    pairs = []
    for _, row in filtered_df.iterrows():
        as_text = str(row["as_text"]).strip()
        ls_text = str(row["ls_text"]).strip()
        if as_text and ls_text:
            pairs.append({
                "source": row["source"],
                "as_text": as_text,
                "ls_text": ls_text
            })
    return pairs

all_pairs = load_filtered_corpus(min_sim=MIN_SIM, max_sim=MAX_SIM)
set_seed(42)
random.shuffle(all_pairs)

# DPO braucht train/val Splits
split_idx = int(0.85 * len(all_pairs))
train_data = all_pairs[:split_idx]
val_data = all_pairs[split_idx:]
print(f"Trainingsdaten: {len(train_data)} Paare | Validierungsdaten: {len(val_data)} Paare")

Globaler Seed auf 42 gesetzt.
Trainingsdaten: 878 Paare | Validierungsdaten: 156 Paare


## 3. Tokenizer & SFT-Baseline laden
Wir initialisieren den Tokenizer mit Quell- und Zielsprache (`de_DE` für Deutsch) und laden die Gewichte aus der SFT-Phase.

In [5]:
print(f"Lade Tokenizer & Modell: {MODEL_NAME}...")
# Tokenizer mit Quell- und Zielsprache initialisieren
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, 
    src_lang="de_DE", 
    tgt_lang="de_DE", 
    model_max_length=MAX_SOURCE_LEN,
    use_fast=False
)

seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Lade SFT-Baseline
if os.path.exists(SFT_MODEL_TEMP_PATH):
    seq2seq_model.load_state_dict(torch.load(SFT_MODEL_TEMP_PATH, map_location=DEVICE))
    print("SFT-Baseline erfolgreich geladen!")
else:
    raise FileNotFoundError(f"Kein SFT-Modell unter {SFT_MODEL_TEMP_PATH} gefunden!")

Lade Tokenizer & Modell: facebook/mbart-large-50...
SFT-Baseline erfolgreich geladen!


## 4. Evaluatoren für Composite Reward laden
Wir laden den BiLSTM Regressor (LS-Grad) und das SBERT-Modell (Semantikerhalt).

In [6]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super(BiLSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        out = self.fc(self.dropout(hidden))
        return self.sigmoid(out)

with open(SYNTHETIC_VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab_data = json.load(f)
    synthetic_stoi = vocab_data.get("stoi", vocab_data)

bilstm_model = BiLSTMRegressor(len(synthetic_stoi), embed_dim=128, hidden_dim=128)
bilstm_model.load_state_dict(torch.load(SYNTHETIC_MODEL_PATH, map_location="cpu"))
bilstm_model.eval()

nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])

def predict_simplicity_score(texts):
    scores = []
    unk_idx = synthetic_stoi.get("<unk>") or synthetic_stoi.get("<UNK>") or 1
    for text in texts:
        doc = nlp(text)
        tokens = [t.text.lower() for t in doc if not t.is_space]
        indices = [synthetic_stoi.get(t, unk_idx) for t in tokens[:150]]
        if len(indices) == 0:
            indices = [0]
        inp_tensor = torch.tensor([indices], dtype=torch.long)
        with torch.no_grad():
            score = bilstm_model(inp_tensor).item()
        scores.append(score)
    return np.array(scores)

sbert_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2", device="cpu")

def predict_semantic_similarity(source_texts, generated_texts):
    emb_src = sbert_model.encode(source_texts, convert_to_tensor=True)
    emb_gen = sbert_model.encode(generated_texts, convert_to_tensor=True)
    return util.cos_sim(emb_src, emb_gen).diagonal().cpu().numpy()

class CompositeRewardEvaluator:
    def __init__(self, w_style=0.5, w_sem=0.5):
        self.w_style = w_style
        self.w_sem = w_sem
        
    def compute_reward(self, source_texts, generated_texts):
        r_style = predict_simplicity_score(generated_texts)
        r_sem = predict_semantic_similarity(source_texts, generated_texts)
        r_sem_norm = np.clip((r_sem + 1.0) / 2.0, 0.0, 1.0)
        total_reward = self.w_style * r_style + self.w_sem * r_sem_norm
        return total_reward, r_style, r_sem_norm

reward_evaluator = CompositeRewardEvaluator(w_style=W_STYLE, w_sem=W_SEM)

## 5. DPO-Präferenzdatensatz erstellen (Offline Generation)
Wir generieren aus dem SFT-Modell 2 Kandidaten per Beam-Search/Sampling, bewerten sie mit der Reward-Funktion und erstellen die `(prompt, chosen, rejected)` Triplets.

In [7]:
def generate_dpo_pairs(data, model, tokenizer, num_samples=None):
    if num_samples is not None:
        data = data[:num_samples]
        
    model.eval()
    dpo_dataset_list = []
    
    # Batch-weise Generierung zur Beschleunigung
    batch_size = 8
    for i in tqdm(range(0, len(data), batch_size), desc="Generiere DPO Paare"):
        batch_items = data[i:i+batch_size]
        src_texts = [item["as_text"] for item in batch_items]
        prompts = ["Übersetze in Leichte Sprache: " + t for t in src_texts]
        
        inputs = tokenizer(prompts, padding=True, truncation=True, max_length=MAX_SOURCE_LEN, return_tensors="pt").to(DEVICE)
        
        # Wir generieren 2 Sequenzen pro Prompt via sampling
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=MAX_TARGET_LEN,
                do_sample=True,
                top_k=50,
                top_p=0.92,
                temperature=0.8,
                num_return_sequences=2
            )
            
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        for idx, src in enumerate(src_texts):
            c1 = decoded[idx * 2]
            c2 = decoded[idx * 2 + 1]
            
            # Paarweise Bewertung
            r1, _, _ = reward_evaluator.compute_reward([src], [c1])
            r2, _, _ = reward_evaluator.compute_reward([src], [c2])
            
            # Zuteilung zu chosen / rejected
            if r1[0] >= r2[0]:
                chosen, rejected = c1, c2
            else:
                chosen, rejected = c2, c1
                
            # Hugging Face DPOTrainer erwartet prompt, chosen, rejected
            dpo_dataset_list.append({
                "prompt": prompts[idx],
                "chosen": chosen,
                "rejected": rejected
            })
            
    return HFDataset.from_list(dpo_dataset_list)

print("Erstelle DPO Trainingsdatensatz...")
dpo_train_dataset = generate_dpo_pairs(train_data, seq2seq_model, tokenizer)
print("Erstelle DPO Validierungsdatensatz...")
dpo_val_dataset = generate_dpo_pairs(val_data, seq2seq_model, tokenizer)

print(f"Präferenz-Daten bereit! Train: {len(dpo_train_dataset)} | Val: {len(dpo_val_dataset)}")

Erstelle DPO Trainingsdatensatz...


Generiere DPO Paare:   0%|          | 0/110 [00:00<?, ?it/s]

Erstelle DPO Validierungsdatensatz...


Generiere DPO Paare:   0%|          | 0/20 [00:00<?, ?it/s]

Präferenz-Daten bereit! Train: 878 | Val: 156


## 6. Training mit dem Hugging Face DPOTrainer
Wir setzen die Trainingsargumente auf und optimieren das Modell unter Verwendung des offiziellen `DPOTrainer`.

In [8]:
# Referenzmodell laden (identisch mit dem SFT-Modell vor DPO)
ref_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
if os.path.exists(SFT_MODEL_TEMP_PATH):
    ref_model.load_state_dict(torch.load(SFT_MODEL_TEMP_PATH, map_location=DEVICE))
ref_model.eval()

training_args = DPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    learning_rate=1e-6,
    warmup_steps=50,
    lr_scheduler_type="linear",
    remove_unused_columns=False, # Wichtig für DPOTrainer!
    run_name="mbart_dpo_trl",
    fp16=torch.cuda.is_available(),
    report_to="none",
    beta=0.1,                    # DPO Temperatur-Parameter
    max_length=MAX_SOURCE_LEN + MAX_TARGET_LEN,
)

dpo_trainer = DPOTrainer(
    model=seq2seq_model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=dpo_train_dataset,
    eval_dataset=dpo_val_dataset,
    processing_class=tokenizer
)

print("Starte DPO-Training mit Hugging Face Trainer...")
dpo_trainer.train()

# Finales Modell abspeichern
dpo_trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Modell erfolgreich gespeichert unter {OUTPUT_DIR}!")

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 23.64 GiB of which 23.50 MiB is free. Process 2675157 has 15.79 GiB memory in use. Process 2678879 has 474.00 MiB memory in use. Including non-PyTorch memory, this process has 7.13 GiB memory in use. Of the allocated memory 6.62 GiB is allocated by PyTorch, and 62.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 7. Evaluierung auf dem unabhängigen Lebenshilfe-Datensatz

In [ ]:
# Lebenshilfe Evaluierung
def evaluate_on_lebenshilfe(model, tokenizer, lh_data_path, reward_evaluator, max_samples=49):
    model.eval()
    with open(lh_data_path, "r", encoding="utf-8") as f:
        lh_data = json.load(f)
        
    as_texts = [item["as_text"] for item in lh_data[:max_samples]]
    ls_ref_texts = [item["ls_text"] for item in lh_data[:max_samples]]
    
    batch_size = 16
    gen_texts = []
    for i in tqdm(range(0, len(as_texts), batch_size), desc="Übersetze Lebenshilfe Datensatz"):
        batch_src = as_texts[i:i+batch_size]
        prompts = ["Übersetze in Leichte Sprache: " + t for t in batch_src]
        inputs = tokenizer(prompts, padding=True, truncation=True, max_length=MAX_SOURCE_LEN, return_tensors="pt").to(DEVICE)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=MAX_TARGET_LEN,
                num_beams=4,
                repetition_penalty=2.5,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        gen_texts.extend(decoded)
        
    tot_reward, r_style, r_sem = reward_evaluator.compute_reward(as_texts, gen_texts)
    sim_to_ref = predict_semantic_similarity(ls_ref_texts, gen_texts)
    sim_to_ref_norm = np.clip((sim_to_ref + 1.0) / 2.0, 0.0, 1.0)
    
    print("\n=================== EVALUIERUNGSERGEBNISSE (LEBENSHILFE) ===================")
    print(f"Ø Simplicity-Einfachheits-Score (R_style):       {r_style.mean():.4f}")
    print(f"Ø SBERT-Ähnlichkeit zur AS-Quelle (R_sem):   {r_sem.mean():.4f}")
    print(f"Ø SBERT-Ähnlichkeit zu echter LS-Referenz:  {sim_to_ref_norm.mean():.4f}")
    print(f"Ø Composite Reward:                        {tot_reward.mean():.4f}")
    print("========================================================================\n")

evaluate_on_lebenshilfe(seq2seq_model, tokenizer, LH_DATASET_PATH, reward_evaluator)